# Notebook 2 — Sentiment EDA
Analyse the FinBERT-scored sentiment output.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid')
TICKERS = ['AAPL', 'MSFT', 'TSLA', 'NVDA', 'GOOGL']

## 1. Load sentiment scores

In [ ]:
sents = {}
for t in TICKERS:
    p = Path(f'../sentiment_scores/{t}_sentiment.csv')
    if p.exists():
        sents[t] = pd.read_csv(p, parse_dates=['date'])
        print(f'{t}: {len(sents[t])} days | mean={sents[t]["mean_score"].mean():.3f}')
    else:
        print(f'[MISSING] {t} — run run_finbert.py on Colab')

## 2. Sentiment score over time

In [ ]:
if sents:
    fig, axes = plt.subplots(len(sents), 1, figsize=(14, 3*len(sents)), sharex=True)
    for ax, (t, df) in zip(axes, sents.items()):
        ax.fill_between(df['date'], df['mean_score'], 0,
                        where=df['mean_score'] >= 0, color='#16a34a', alpha=0.5, label='Positive')
        ax.fill_between(df['date'], df['mean_score'], 0,
                        where=df['mean_score'] < 0, color='#dc2626', alpha=0.5, label='Negative')
        ax.axhline(0, color='black', linewidth=0.7)
        ax.set_ylabel(t, fontsize=10)
        ax.set_ylim(-1, 1)
    plt.suptitle('FinBERT Daily Sentiment Score', fontsize=12)
    plt.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

## 3. Sentiment vs next-day price return

In [ ]:
import warnings
warnings.filterwarnings('ignore')

for t in list(sents.keys())[:3]:
    price_path = Path(f'../raw_data/prices/{t}_2018_2024.csv')
    if not price_path.exists():
        continue
    price = pd.read_csv(price_path, parse_dates=['Date'])[['Date', 'Close']].rename(columns={'Date':'date'})
    price['next_return'] = price['Close'].pct_change().shift(-1)
    merged = sents[t].merge(price[['date','next_return']], on='date')
    merged = merged.dropna()

    corr = merged['mean_score'].corr(merged['next_return'])
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.scatter(merged['mean_score'], merged['next_return'], alpha=0.3, s=10, c='#2563eb')
    ax.axhline(0, color='gray', lw=0.5)
    ax.axvline(0, color='gray', lw=0.5)
    ax.set_xlabel('FinBERT Sentiment Score')
    ax.set_ylabel('Next-Day Return')
    ax.set_title(f'{t} | Pearson r = {corr:.3f}')
    plt.tight_layout()
    plt.show()

## 4. Headline coverage statistics

In [ ]:
if sents:
    total_stats = []
    for t, df in sents.items():
        total_stats.append({
            'Ticker': t,
            'Total Trading Days': len(df),
            'Total Headlines': int(df['headline_count'].sum()),
            'Avg Headlines/Day': df['headline_count'].mean(),
            'Days With No News': int((df['headline_count'] == 0).sum()),
            'Mean Sentiment': df['mean_score'].mean(),
            'Std Sentiment': df['mean_score'].std(),
        })
    pd.DataFrame(total_stats).set_index('Ticker').round(3)